# 🔥 Forest Fire Prediction — EDA & Model Building
> **Dataset:** Algerian Forest Fires (UCI / Kaggle)  
> **Tasks:** Binary Classification (Fire / No-Fire) + FWI Regression


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.facecolor'] = '#1a1a1a'
plt.rcParams['axes.facecolor']   = '#2d2d2d'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
print('Libraries loaded ✅')

## 1. Load & Clean Dataset

In [ ]:
from train_models import load_algerian_csv
df = load_algerian_csv('../dataset/Algerian_forest_fires_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['Classes'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
features  = ['Temperature','RH','Ws','Rain','FFMC','DMC','DC','ISI','BUI','FWI']
colors    = ['#FF4500','#FF7B00']
for ax, feat in zip(axes.flat, features):
    for cls, col in zip([0,1], colors):
        subset = df[df['Classes'] == cls][feat]
        ax.hist(subset, bins=20, alpha=0.7, color=col, label=['No Fire','Fire'][cls])
    ax.set_title(feat, color='white')
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Class', fontsize=14, color='#FF7B00')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='YlOrRd', linewidths=0.5, linecolor='#1a1a1a')
plt.title('Feature Correlation Heatmap', color='#FF7B00', fontsize=14)
plt.show()

## 3. Model Training & Evaluation

In [ ]:
import sys; sys.path.insert(0, '..')
from train_models import train_tabular, train_image_model
clf, reg, scaler = train_tabular('../dataset/Algerian_forest_fires_dataset.csv')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from train_models import load_algerian_csv, TABULAR_FEATURES
from sklearn.model_selection import train_test_split

df = load_algerian_csv('../dataset/Algerian_forest_fires_dataset.csv')
X  = scaler.transform(df[TABULAR_FEATURES].values)
y  = df['Classes'].values
_, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_te, clf.predict(X_te),
    display_labels=['No Fire','Fire'],
    cmap='YlOrRd', ax=ax)
ax.set_title('Classifier Confusion Matrix', color='#FF7B00')
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
fi   = clf.feature_importances_
feat = TABULAR_FEATURES
idx  = np.argsort(fi)[::-1]

plt.figure(figsize=(10, 5))
plt.bar([feat[i] for i in idx], fi[idx], color=['#FF4500' if i < 3 else '#FF7B00' for i in range(len(idx))])
plt.title('Feature Importance (Gradient Boosting Classifier)', color='#FF7B00')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 4. Image Model Training

In [ ]:
img_clf = train_image_model('../dataset/images')
if img_clf:
    print('Image model trained successfully ✅')
else:
    print('Image folder not found — add images to dataset/images/fire/ and dataset/images/nofire/')